<a href="https://colab.research.google.com/github/GHROTH-L/-ai-ml-training-/blob/main/Predicting_Irrigation_Need.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [24]:
import pandas as pd
import numpy as np
import io
import matplotlib.pyplot as plt
import seaborn as sns #畫圖使用
%matplotlib inline
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression
from xgboost import XGBRegressor , XGBClassifier
from sklearn.preprocessing import OneHotEncoder , LabelEncoder, TargetEncoder
from scipy.stats import f_oneway
from sklearn.model_selection import train_test_split , cross_val_score
from sklearn.metrics import accuracy_score, root_mean_squared_error , balanced_accuracy_score, f1_score, classification_report

#將dataframe 下載下來
from google.colab import files

In [2]:
from google.colab import files
# 上傳
uploaded = files.upload()  #for train
uploaded2 = files.upload()  #for test

Saving test.csv to test.csv


Saving train.csv to train.csv


In [33]:
train = pd.read_csv('train.csv')   # 最一開始
test = pd.read_csv('test.csv')

In [34]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 630000 entries, 0 to 629999
Data columns (total 21 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   id                       630000 non-null  int64  
 1   Soil_Type                630000 non-null  object 
 2   Soil_pH                  630000 non-null  float64
 3   Soil_Moisture            630000 non-null  float64
 4   Organic_Carbon           630000 non-null  float64
 5   Electrical_Conductivity  630000 non-null  float64
 6   Temperature_C            630000 non-null  float64
 7   Humidity                 630000 non-null  float64
 8   Rainfall_mm              630000 non-null  float64
 9   Sunlight_Hours           630000 non-null  float64
 10  Wind_Speed_kmh           630000 non-null  float64
 11  Crop_Type                630000 non-null  object 
 12  Crop_Growth_Stage        630000 non-null  object 
 13  Season                   630000 non-null  object 
 14  Irri

In [35]:
target_col = 'Irrigation_Need'

X = train.drop(columns=target_col)
y = train[target_col]


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

In [36]:
obj_cols = [
    'Soil_Type',
    'Crop_Type',
    'Crop_Growth_Stage',
    'Season',
    'Irrigation_Type',
    'Water_Source',
    'Mulching_Used',
    'Region'
]


X_train = pd.get_dummies(X_train, columns=obj_cols, drop_first=True)
X_test = pd.get_dummies(X_test, columns=obj_cols, drop_first=True)

X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

base_features = [
    'Soil_pH',
    'Soil_Moisture',
    'Organic_Carbon',
    'Electrical_Conductivity',
    'Temperature_C',
    'Humidity',
    'Rainfall_mm',
    'Sunlight_Hours',
    'Wind_Speed_kmh'
]

dummy_prefixes = ['Soil_Type_', 'Crop_Type_', 'Crop_Growth_Stage_']

dummy_features = [
    col for col in X_train.columns
    if any(col.startswith(prefix) for prefix in dummy_prefixes)
]

features = base_features + dummy_features


In [42]:
print(X_train[features].shape)

(504000, 20)


In [45]:
lr = LogisticRegression(
    max_iter=2000,
    random_state=42
)

rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

xgb = XGBClassifier(
    n_estimators=1000,
    learning_rate=0.03,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1,
    random_state=42,
    eval_metric='mlogloss'
)
gbr = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.05,
    random_state=42
)

# fit
#lr.fit(X_train[features], y_train_enc)
#rf.fit(X_train[features], y_train_enc)
xgb.fit(X_train[features], y_train_enc)
#gbr.fit(X_train[features], y_train_enc)

# predict
#pred_lr = lr.predict(X_test[features])
#pred_rf = rf.predict(X_test[features])
pred_xgb = xgb.predict(X_test[features])
#pred_gbr = gbr.predict(X_test[features])


In [47]:
#print("LR balanced acc:", balanced_accuracy_score(y_test_enc, pred_lr))
#print("RF balanced acc:", balanced_accuracy_score(y_test_enc, pred_rf))
print("XGB balanced acc:", balanced_accuracy_score(y_test_enc, pred_xgb))
#print("GBR balanced acc:", balanced_accuracy_score(y_test_enc, pred_gbr))

#print("LR macro F1:", f1_score(y_test_enc, pred_lr, average='macro'))
#print("RF macro F1:", f1_score(y_test_enc, pred_rf, average='macro'))
print("XGB macro F1:", f1_score(y_test_enc, pred_xgb, average='macro'))
#print("GBR macro F1:", f1_score(y_test_enc, pred_gbr, average='macro'))

XGB balanced acc: 0.8884809381365533
XGB macro F1: 0.8804160020988882


In [48]:
test_enc = test.copy()

# 如果 test 裡有 id 欄位，先留著做 submission
test_ids = test_enc['id']   # 若不是 id，改成你的實際欄位名

# 做和 train 一樣的 dummy encoding
test_enc = pd.get_dummies(test_enc, columns=obj_cols, drop_first=True)

# 跟訓練資料欄位對齊
# 這裡要用 X_train 的全部欄位來對齊，不是只對齊 features
test_enc = test_enc.reindex(columns=X_train.columns, fill_value=0)

# 預測（先得到編碼後的類別）
pred_test_enc = xgb.predict(test_enc[features])

# 如果你前面有用 LabelEncoder，就要轉回原始類別名稱
pred_test = le.inverse_transform(pred_test_enc)

# 做 submission
submission = pd.DataFrame({
    'id': test_ids,
    'Irrigation_Need': pred_test
})

submission.to_csv('submission.csv', index=False)
print(submission.head())

       id Irrigation_Need
0  630000             Low
1  630001          Medium
2  630002             Low
3  630003             Low
4  630004             Low


In [49]:

files.download('submission.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>